# Electricity meta-set creation

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
DATA_PATH = PROJECT_ROOT / 'data' / 'electricity' / 'processed' / 'electricity_hourly.csv'
SAVE_DIR = PROJECT_ROOT / 'data' / 'electricity' / 'meta_sets'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

elec_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(elec_df)} rows, {elec_df['id'].nunique()} series, "
      f"days_from_start in [{elec_df['days_from_start'].min()}, {elec_df['days_from_start'].max()}]")

In [ ]:
elec_df['log_power_usage'] = np.log1p(elec_df['power_usage'])
print(f"power_usage:      min={elec_df['power_usage'].min():.4f}  max={elec_df['power_usage'].max():.4f}  "
      f"frac_zero={(elec_df['power_usage']==0).mean():.4f}")
print(f"log_power_usage:  min={elec_df['log_power_usage'].min():.4f}  max={elec_df['log_power_usage'].max():.4f}")

## Split scheme

In [ ]:
SEED = 42
ID_COL = 'id'          # meter id; renamed to traj_id at save time
DAY_COL = 'days_from_start'

TRAIN_START, TRAIN_END = 1096, 1314
VAL_START, VAL_END = 1308, 1315
TEST_START, TEST_END = 1309, 1316

TRAIN_ITEM_COUNT = 20
VAL_ITEM_COUNT = 85
TEST_ITEM_COUNT = 84 

DATE_TAG = datetime.now().strftime('%d%m')

In [ ]:
np.random.seed(SEED)

test_pool = set(elec_df.loc[(elec_df[DAY_COL] >= TEST_START) & (elec_df[DAY_COL] <= TEST_END), ID_COL].unique())
val_pool = set(elec_df.loc[(elec_df[DAY_COL] >= VAL_START) & (elec_df[DAY_COL] <= VAL_END), ID_COL].unique())
train_pool_period = set(elec_df.loc[(elec_df[DAY_COL] >= TRAIN_START) & (elec_df[DAY_COL] <= TRAIN_END), ID_COL].unique())

if len(test_pool) < TEST_ITEM_COUNT:
    raise ValueError(f"Test pool too small: {len(test_pool)} < {TEST_ITEM_COUNT}")
test_ids = np.random.choice(sorted(test_pool), size=TEST_ITEM_COUNT, replace=False)

val_pool_avail = val_pool - set(test_ids)
if len(val_pool_avail) < VAL_ITEM_COUNT:
    raise ValueError(f"Val pool too small after excluding test series: {len(val_pool_avail)} < {VAL_ITEM_COUNT}")
val_ids = np.random.choice(sorted(val_pool_avail), size=VAL_ITEM_COUNT, replace=False)

train_pool = train_pool_period - (set(val_ids) | set(test_ids))

shuffled_train_pool = np.random.permutation(sorted(train_pool)).tolist()
if len(shuffled_train_pool) < TRAIN_ITEM_COUNT:
    raise ValueError(f"Train pool too small: {len(shuffled_train_pool)} < {TRAIN_ITEM_COUNT}")

In [ ]:
selected_ids = shuffled_train_pool[:TRAIN_ITEM_COUNT]
train_df = elec_df[
    (elec_df[ID_COL].isin(selected_ids)) &
    (elec_df[DAY_COL] >= TRAIN_START) & (elec_df[DAY_COL] <= TRAIN_END)
].copy()

n_actual = train_df[ID_COL].nunique()
if n_actual != TRAIN_ITEM_COUNT:
    raise ValueError(f"Expected {TRAIN_ITEM_COUNT} series with data in train period, got {n_actual}")

train_df = train_df.rename(columns={ID_COL: 'traj_id', 't': 'time_idx'})
save_path = SAVE_DIR / f'Seed{SEED}_electricity_seriesBased_meta_train_cumulative_{TRAIN_ITEM_COUNT}ids_{DATE_TAG}.pkl'
train_df.to_pickle(save_path)

In [ ]:
val_df = elec_df[
    (elec_df[ID_COL].isin(val_ids)) &
    (elec_df[DAY_COL] >= VAL_START) & (elec_df[DAY_COL] <= VAL_END)
].copy()
n_actual = val_df[ID_COL].nunique()
if n_actual != VAL_ITEM_COUNT:
    raise ValueError(f"Val: expected {VAL_ITEM_COUNT} series, got {n_actual}")

val_df = val_df.rename(columns={ID_COL: 'traj_id', 't': 'time_idx'})
save_path = SAVE_DIR / f'Seed{SEED}_electricity_seriesBased_meta_val_{VAL_ITEM_COUNT}ids_{DATE_TAG}.pkl'
val_df.to_pickle(save_path)

## New test set (200 series)

In [ ]:
NEW_TEST_ITEM_COUNT = 200
NEAR_DEAD_THRESHOLD = 10  

train_used = set(pd.read_pickle(SAVE_DIR / 'Seed42_electricity_seriesBased_meta_train_cumulative_20ids_0508.pkl')['traj_id'].unique())
val_used = set(pd.read_pickle(SAVE_DIR / 'Seed42_electricity_seriesBased_meta_val_85ids_0508.pkl')['traj_id'].unique())
candidate_pool = sorted(set(elec_df[ID_COL].unique()) - train_used - val_used)

In [ ]:
test_window_all = elec_df[
    (elec_df[DAY_COL] >= TEST_START) & (elec_df[DAY_COL] <= TEST_END)
    & (elec_df[ID_COL].isin(candidate_pool))
]
query_volume = (test_window_all.sort_values('t').groupby(ID_COL).tail(24)
                .groupby(ID_COL)['power_usage'].sum())
near_dead = set(query_volume[query_volume < NEAR_DEAD_THRESHOLD].index)

clean_pool = sorted(set(candidate_pool) - near_dead)

np.random.seed(SEED)
new_test_ids = sorted(np.random.choice(clean_pool, size=NEW_TEST_ITEM_COUNT, replace=False))

In [ ]:
new_test_df = elec_df[
    (elec_df[ID_COL].isin(new_test_ids)) & (elec_df[DAY_COL] >= TEST_START) & (elec_df[DAY_COL] <= TEST_END)
].copy()
n_actual = new_test_df[ID_COL].nunique()
if n_actual != NEW_TEST_ITEM_COUNT:
    raise ValueError(f"Test: expected {NEW_TEST_ITEM_COUNT} series, got {n_actual}")
assert not (set(new_test_ids) & train_used), "leakage vs train!"
assert not (set(new_test_ids) & val_used), "leakage vs val!"
assert not (set(new_test_ids) & near_dead), "near-dead series slipped in!"

new_test_df = new_test_df.rename(columns={ID_COL: 'traj_id', 't': 'time_idx'})
save_path = SAVE_DIR / f'Seed{SEED}_electricity_seriesBased_meta_test_{NEW_TEST_ITEM_COUNT}ids_volfilt{DATE_TAG}.pkl'
new_test_df.to_pickle(save_path)